In [1]:
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
import matlab.engine
import numpy as np
import plotly.graph_objects as go
import pandas as pd
import json
import os

# Start MATLAB Engine
eng = matlab.engine.start_matlab()

# Load Simulink Model
model_path = 'Intermodal_simulation_IBonly_Dash.slx'
eng.load_system(model_path, nargout=0)

# Dash App Setup
app = dash.Dash(__name__)


def load_saved_figure1():
    try:
        with open("saved_figure1.json", "r") as f:
            fig1_json = f.read()
        return go.Figure(json.loads(fig1_json))
    except FileNotFoundError:
        return go.Figure()  # Return an empty figure if no file exists

def load_saved_figure2():    
    try:
        with open("saved_figure2.json", "r") as f:
            fig2_json = f.read()
        return go.Figure(json.loads(fig2_json))
    except FileNotFoundError:
        return go.Figure()  # Return an empty figure if no file exists


# Set initial figure when the app loads
initial_figure1 = load_saved_figure1()
initial_figure2 = load_saved_figure2()


app.layout = html.Div([
    html.H1("Intermodal Facility Simulation", style={'fontFamily': 'Arial, Helvetica, sans-serif'}),

    html.Div([
        html.Label("Average Hostler Refueling Time:", style={'fontFamily': 'Arial, Helvetica, sans-serif'}),
        dcc.Input(id="input-param-1", type="number", value=90, step=0.1)
    ], style={'display': 'grid', 'grid-template-columns': '300px 75px'}),
    
    html.Div([
        html.Label("Number of Hostler Charging Stations:", style={'fontFamily': 'Arial, Helvetica, sans-serif'}),
        dcc.Input(id="input-param-3", type="number", value=8, step=1)
    ], style={'display': 'grid', 'grid-template-columns': '300px 75px'}),

    html.Div([
        html.Label("Average Hostler Move Cycle Time:", style={'fontFamily': 'Arial, Helvetica, sans-serif'}),
        dcc.Input(id="input-param-2", type="number", value=20, step=0.1)
    ], style={'display': 'grid', 'grid-template-columns': '300px 75px'}),

    html.Br(),
    
    html.Div([
        html.Button("Run Simulation", id="run-button", n_clicks=0, 
                    style={'fontSize': '18px', 'padding': '15px 30px', 'width': '200px', 'height': '60px', 'fontFamily': 'Arial, Helvetica, sans-serif'})], 
             style = {'textAlign': 'left'}),

    dcc.Graph(id="sim-graph-1", figure=initial_figure1),
    dcc.Graph(id="sim-graph-2", figure=initial_figure2),

    html.H3("Simulation Summary Statistics", style={'fontFamily': 'Arial, Helvetica, sans-serif', 'textAlign': 'center', 'margin-bottom': '5px'}),
    dash_table.DataTable(id="summary-table-1",
                         columns=[{"name": i, "id": i} for i in ["Statistic", "Average Charger Wait (minutes)", "Containers in Staging Area"]],
                         style_table = {'width': '75%', 'margin': 'auto'}),

    #html.H3("Average Charger Wait (minutes)", style={'fontFamily': 'Arial, Helvetica, sans-serif', 'textAlign': 'center', 'margin-bottom': '5px'}),
    #dash_table.DataTable(id="summary-table-1", 
     #                    columns=[{"name": i, "id": i} for i in ["Statistic", "Value"]],
     #                    style_table={'width': '30%', 'margin': 'auto'}),

    #html.Br(),

    #html.H3("Containers in Staging Area", style={'fontFamily': 'Arial, Helvetica, sans-serif', 'textAlign': 'center', 'margin-bottom': '5px'}),
    #dash_table.DataTable(id="summary-table-2", 
    #                     columns=[{"name": i, "id": i} for i in ["Statistic", "Value"]],
    #                     style_table={'width': '30%', 'margin': 'auto'}),
    
        html.Button("Quit MATLAB", id="quit-matlab-button", n_clicks=0, style={'margin-left': '10px', 'backgroundColor': 'red', 'color': 'white'}),

])


@app.callback(
    [Output("sim-graph-1", "figure"),
     Output("sim-graph-2", "figure"),
     Output("summary-table-1", "data")#,
     #Output("summary-table-2", "data")
    ],
    Input("run-button", "n_clicks"),
    [State("input-param-1", "value"),
     State("input-param-2", "value"),
     State("input-param-3", "value")],
    prevent_initial_call=True
)
def run_simulation(n_clicks, input_value_1, input_value_2, input_value_3):
    try:
        model_name = 'Intermodal_simulation_IBonly_Dash'

        simulation_time = 10000  # Example: User-defined run time
        eng.set_param('Intermodal_simulation_IBonly_Dash', 'StopTime', str(simulation_time), nargout=0)# Set simulation run time dynamically
        
        # Set input parameters in the model
        eng.set_param(f"{model_name}/Hostler Charging/meanChargingTime", 'Value', str(input_value_1), nargout=0)
        eng.set_param(f"{model_name}/Hostler Container Move/meanServiceTime", 'Value', str(input_value_2), nargout=0)
        eng.set_param(f"{model_name}/Hostler Charging/Hostler Charging", 'Capacity', str(input_value_3), nargout=0)

        # Run the simulation
        eng.eval("out = sim('Intermodal_simulation_IBonly_Dash')", nargout=0)
        eng.eval("data = struct(out)", nargout=0)
        data = eng.workspace['data']
        #print(data)

        # Access Charger_Wait Data
        charger_wait_data = data['Data']['Charger_Wait']
        time_data_1 = np.array(charger_wait_data['time']).flatten()
        values_1 = np.array(charger_wait_data['signals']['values']).flatten()

        # Access Staging data"
        other_signal_data = data['Data']['Staging_IBonly']
        time_data_2 = np.array(other_signal_data['time']).flatten()
        values_2 = np.array(other_signal_data['signals']['values']).flatten()
        
        # Create first figure
        fig1 = go.Figure(initial_figure1)
        fig1.add_trace(go.Scatter(x=time_data_1, y=values_1, mode="lines", name="Charger Wait Time (Simulation Run)"))
        fig1.update_layout(title="Average Charger Wait Time", title_x=0.4, margin=dict(t=30), xaxis_title="Simulation Time (minutes)", yaxis_title="Wait Time (minutes)")

        # Save the figure as JSON
        #fig1_json = fig1.to_json()
        #with open("saved_figure1.json", "w") as f:
        #    f.write(fig1_json)

        # Create second figure
        fig2 = go.Figure(initial_figure2)
        fig2.add_trace(go.Scatter(x=time_data_2, y=values_2, mode="lines", name="Staging Spaces (Simulation Run)"))
        fig2.update_layout(title="Containers in Staging Area", title_x=0.4, margin=dict(t=30), xaxis_title="Simulation Time (minutes)", yaxis_title="Staging Spaces in Use")

        # Save the figure as JSON
        #fig2_json = fig2.to_json()
        #with open("saved_figure2.json", "w") as f:
        #   f.write(fig2_json)

        # Compute Summary Statistics
        summary_stats_1 = pd.DataFrame({
            "Statistic": ["Mean", "Median", "Min", "Max", "Std Dev"],
            "Average Charger Wait (minutes)": [np.mean(values_1), np.median(values_1), np.min(values_1), np.max(values_1), np.std(values_1)],
            "Containers in Staging Area": [np.mean(values_2), np.median(values_2), np.min(values_2), np.max(values_2), np.std(values_2)]
        })

       # summary_stats_2 = pd.DataFrame({
       #     "Statistic": ["Mean", "Median", "Min", "Max", "Std Dev"],
       #     "Value": [np.mean(values_2), np.median(values_2), np.min(values_2), np.max(values_2), np.std(values_2)]
       # })

        # Round values to 2 decimal places
        summary_stats_1["Average Charger Wait (minutes)"] = summary_stats_1["Average Charger Wait (minutes)"].round(2)
        summary_stats_1["Containers in Staging Area"] = summary_stats_1["Containers in Staging Area"].round(0)

        return fig1, fig2, summary_stats_1.to_dict("records")#, summary_stats_2.to_dict("records")

    except Exception as e:
        print(f"Error during simulation: {e}")
        return go.Figure(), go.Figure(), None, None  # Return empty figures in case of error

    #finally:
    #    if eng is not None:
    #        eng.quit()

    def quit_matlab(n_clicks):
        global eng
        if eng is not None:
            eng.quit()
            eng = None  # Reset engine to prevent further calls
        return True  # Disable the button to prevent multiple clicks

if __name__ == "__main__":
    app.run_server(debug=True)

OSError: Address 'http://127.0.0.1:8050' already in use.
    Try passing a different port to run_server.

In [1]:
import matlab.engine

imports_to_test = [

 "time",
 "pandas",
 "geopandas",
 "networkx",
 "gurobipy",
 "pyproj",
 "random",
 "dash_bootstrap_components",
 "dash_leaflet",
 "pickle",
 "numpy as np"
 "dash",
 "concurrent",
 "plotly",
 "pyproj",
 "random",
 "matplotlib",
 "shapely",
 "geopy",
 "pyproj",
 "plotly",
 "time",
 "ast",
"json",
"itertools"
"os",

]

for lib in imports_to_test:
    try:
        exec(f"import {lib}")
        print(f" {lib} imported successfully")
    except Exception as e:
        print(f" {lib} failed to import: {e}")

 time imported successfully
 pandas imported successfully
 geopandas imported successfully
 networkx imported successfully
 gurobipy imported successfully
 pyproj imported successfully
 random imported successfully
 dash_bootstrap_components imported successfully
 dash_leaflet imported successfully
 pickle imported successfully
 numpy as npdash imported successfully
 concurrent imported successfully
 plotly imported successfully
 pyproj imported successfully
 random imported successfully
 matplotlib imported successfully
 shapely imported successfully
 geopy imported successfully
 pyproj imported successfully
 plotly imported successfully
 time imported successfully
 ast imported successfully
 json imported successfully
 itertoolsos failed to import: No module named 'itertoolsos'
